# Raz (2025) Replication Data — Usage Notebook

This notebook walks through loading and using the data package built for the
NUS BZD6004 group replication of:

> Raz, I. T. (2025). *Soil Heterogeneity, Social Learning, and the Formation
> of Close-Knit Communities*, JPE 133(8), 2643-2691.

Run all cells from top to bottom. The expected file layout next to this
notebook is:

```
final_delivery/
├── README.md
├── notebooks/
│   └── demo_usage.ipynb          ← you are here
├── figures/                       ← populated by this notebook
└── data/
    ├── CountyLevelData.csv
    ├── SoilHeterogeneityIndex.csv
    ├── GeoClimaticControls.csv
    ├── SmoothLocationControls.csv
    ├── FertilizersData.csv
    ├── WheatShareData.csv
    ├── WheatProductionData.csv
    ├── AgriculturalDiversityData.csv
    ├── SlaveShareData.csv
    ├── ReligiousDiversityData.csv
    ├── FarmGiniData.csv
    ├── BirthPlaceDiversityData.csv
    └── countypres_2000-2024.tab
```


## 0 · Environment

In [ ]:
import sys, os
from pathlib import Path

# Package root = parent of this notebook's parent
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
FIGS = ROOT / "figures"
FIGS.mkdir(exist_ok=True)
print("ROOT:", ROOT)
print("DATA folder contents:")
for p in sorted(DATA.iterdir()):
    print(" ", p.name)


## 1 · Load the master panel

`CountyLevelData.csv` is the single file you'll likely work with most. It's
a county-year panel with 30 columns. Join key = `gisjoin × year`.

In [ ]:
import pandas as pd

panel = pd.read_csv(DATA / "CountyLevelData.csv")
print("Shape:", panel.shape)
print("Counties:", panel["gisjoin"].nunique())
print("Years:", sorted(panel["year"].unique()))
panel.head()


In [ ]:
# Quick look at variable coverage (non-NA counts)
pd.DataFrame({
    "non_na": panel.notna().sum(),
    "dtype": panel.dtypes.astype(str),
})


## 2 · Example: SHI → Fertilizer adoption (Table 5 style)

The paper's Table 5 regresses the *growth* of fertilizer adoption on SHI
with state-by-year fixed effects and county-level controls, clustered at
grid cells. As a starting point we run a *level* regression on 1920 with
state FE and clustered-by-state SEs. To match the paper fully you would
(a) compute the growth rate 1910→1930 as the dependent variable, and
(b) use arbitrary grid-cell clusters per Bester et al. (2011) instead of
state clusters.

In [ ]:
import statsmodels.api as sm

CONTROL_COLS = [
    "mean_elevation_m", "mean_slope_deg",
    "mean_annual_temp_c", "mean_annual_precip_mm",
    "mean_flow_accum", "river_density",
    "centroid_lat", "centroid_lon", "lat_sq", "lon_sq", "lat_x_lon",
]

def ols_with_state_fe(df, y_col, ctrls=CONTROL_COLS):
    d = df.dropna(subset=["shi", y_col, *ctrls, "state"]).copy()
    X = pd.concat(
        [
            pd.Series(1.0, index=d.index, name="const"),
            d[["shi", *ctrls]],
            pd.get_dummies(d["state"], prefix="state", drop_first=True, dtype=float),
        ],
        axis=1,
    )
    y = d[y_col].astype(float)
    return sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": d["state"]})

d1920 = panel[panel["year"] == 1920]
fit_fert = ols_with_state_fe(d1920, "share_farms_reporting_fert")
print("SHI coef on share_farms_reporting_fert (1920, state FE):")
print(f"  {fit_fert.params['shi']:+.4f}  (SE={fit_fert.bse['shi']:.4f}, p={fit_fert.pvalues['shi']:.3f})")
print(f"  n = {int(fit_fert.nobs)}")


## 3 · Example: SHI → Agricultural diversity (Table 4 style)

Same specification but dependent variable is `ag_diversity_index` = 1 −
Herfindahl over crop acreages. Paper finds a strongly positive effect.

In [ ]:
fit_div = ols_with_state_fe(
    d1920, "ag_diversity_index",
    ctrls=[c for c in CONTROL_COLS if c not in ("mean_flow_accum", "river_density")],
)
print("SHI coef on ag_diversity_index (1920, state FE):")
print(f"  {fit_div.params['shi']:+.4f}  (SE={fit_div.bse['shi']:.4f}, p={fit_div.pvalues['shi']:.3f})")
print(f"  n = {int(fit_div.nobs)}")


## 4 · Figure 1 style map: county-level SHI across CONUS

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

# The 1940 county shapefile is not shipped in this zip to keep it small.
# It is the NHGIS 1940 tl2008 shapefile. The script that built this package
# saved it as a parquet file one level up from the delivery folder — adjust
# the path if you copied the notebook elsewhere.
county_pq = ROOT.parent / "data_collection" / "processed" / "county_1940_albers.parquet"
if county_pq.exists():
    county = gpd.read_parquet(county_pq)
    shi = panel[["gisjoin", "shi"]].drop_duplicates("gisjoin")
    gdf = county.merge(shi, left_on="GISJOIN", right_on="gisjoin", how="left")
    gdf = gdf.cx[-2.5e6:2.5e6, -2e6:3.5e6]  # CONUS clip

    fig, ax = plt.subplots(figsize=(12, 7))
    gdf.plot(
        column="shi", cmap="viridis", linewidth=0.1,
        edgecolor="white", legend=True,
        legend_kwds={"label": "Soil Heterogeneity Index (1 − HHI)", "shrink": 0.6},
        ax=ax, missing_kwds={"color": "#eeeeee"},
    )
    ax.set_title("Figure 1 (replicated): County SHI — CONUS", fontsize=13)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(FIGS / "figure1_shi_map.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("County shapefile parquet not found at", county_pq)
    print("Skipping map. Image pre-built at figures/figure1_shi_map.png")


## 5 · SHI × Fertilizer scatter (1920)

In [ ]:
import numpy as np

d = panel[(panel["year"] == 1920)].dropna(subset=["shi", "share_farms_reporting_fert"])
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(d["shi"], d["share_farms_reporting_fert"], s=3, alpha=0.3, color="#1f77b4")
bins = np.linspace(d["shi"].min(), d["shi"].max(), 25)
d = d.assign(_bin=pd.cut(d["shi"], bins))
gm = d.groupby("_bin", observed=True).agg(
    x=("shi", "mean"), y=("share_farms_reporting_fert", "mean")
)
ax.plot(gm["x"], gm["y"], "-o", color="crimson", label="binned mean")
ax.set_xlabel("SHI")
ax.set_ylabel("Share of farms reporting fertilizer")
ax.set_title("SHI vs Fertilizer adoption (US counties, 1920)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "shi_vs_fertilizer_1920.png", dpi=180)
plt.show()


## 6 · Variable trends over time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
plots = [
    (axes[0, 0], "share_farms_reporting_fert", "Fertilizer adoption"),
    (axes[0, 1], "wheat_share_of_farmland", "Wheat share of farmland"),
    (axes[1, 0], "ag_diversity_index", "Ag diversity (1 − HHI)"),
    (axes[1, 1], "religious_diversity_index", "Religious diversity (1 − HHI)"),
]
for ax, col, label in plots:
    s = panel.groupby("year")[col].mean()
    ax.plot(s.index, s.values, "-o", color="#2ca02c")
    ax.set_title(label)
    ax.set_xlabel("Year")
    ax.grid(alpha=0.3)
plt.suptitle("County-mean values by decade", fontsize=13)
plt.tight_layout()
plt.savefig(FIGS / "variable_trends.png", dpi=180)
plt.show()


## 7 · Future: plugging in IPUMS-dependent columns

When the IPUMS full-count restricted application is approved, the team will
construct the Local Name Index (`LNI`), Strength of Family Ties (`SFT`),
share of farmers, and full-count Birth Place Diversity from IPUMS individual
microdata. Those can be merged into `CountyLevelData` like this:

```python
ipums_county_vars = (
    pd.read_csv("lni_sft_from_ipums.csv")      # your own build
      .rename(columns={"GISJOIN": "gisjoin"})
)
panel_full = panel.merge(
    ipums_county_vars,
    on=["gisjoin", "year"],
    how="left",
)
```

At that point the main paper regressions (Tables 1-3, 7, 8) become runnable
on the merged panel. The Census Linking Project crosswalks in
`../data_collection/raw/census_linking/` (kept outside this delivery zip)
let you build the linked-census outputs (Tables 2-4, 6; Figure 3).

---

**End of demo notebook.** For a detailed manifest of how each file was built,
see `README.md` and the build scripts at `../data_collection/scripts/`.
